# 🇮🇩 TaskGen Indonesia — Colab satu sel

Notebook ringkas: jalankan satu sel di bawah, atur opsi, lalu tekan **Generate**. Bahasa keluaran dikunci ke `id` (Indonesia).

Semua pertanyaan disimpan dalam satu file `MyDrive/TaskGen-Indonesia/task_id.json` berbentuk JSON array. Setiap task hanya menyimpan enam field inti: `prompt`, `domain`, `subdomain`, `difficulty`, `language`, dan `model`. Temperature default adalah `1.0`. Deduplikasi opsional memeriksa prompt lama dan prompt dalam run yang sama.

**Catatan:** sel ini mengunduh kode dari branch `main` dan membangun Rust binary, sehingga run pertama mungkin membutuhkan waktu.


In [2]:
# 🇮🇩 TaskGen Indonesia — satu sel untuk semua
# Jalankan sel ini sekali. Setelah UI muncul, atur opsi lalu tekan Generate.

import os, sys, json, re, hashlib, subprocess, shutil, time
from pathlib import Path

# Dependensi UI
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "ipywidgets", "requests"], check=True)
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive, userdata

# Google Drive
print("Menghubungkan Google Drive...")
drive.mount('/content/drive', force_remount=False)
DRIVE_DIR = Path('/content/drive/MyDrive/TaskGen-Indonesia')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = DRIVE_DIR / 'task_id.json'

# Ambil API key dari Colab Secrets atau minta secara tersembunyi
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = None
if not API_KEY:
    from getpass import getpass
    API_KEY = getpass('Masukkan Gemini API key (input disembunyikan): ').strip()
if not API_KEY:
    raise ValueError('Gemini API key kosong. Tambahkan GEMINI_API_KEY di Colab Secrets.')
os.environ['GEMINI_API_KEY'] = API_KEY

# Gunakan main agar notebook memakai kode terbaru yang sudah digabung
REPO_URL = 'https://github.com/IND-NV/taskgen-id.git'
BRANCH = 'main'
REPO_DIR = Path('/content/taskgen-id')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
print('Mengunduh TaskGen dari branch:', BRANCH)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

if shutil.which('cargo') is None:
    print('Memasang Rust toolchain...')
    subprocess.run(['bash', '-lc', "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y"], check=True)
    os.environ['PATH'] = '/root/.cargo/bin:' + os.environ.get('PATH', '')
if shutil.which('cargo') is None:
    raise RuntimeError('Cargo tidak ditemukan setelah instalasi Rust.')
print('Membangun TaskGen (pertama kali bisa memerlukan beberapa menit)...')
subprocess.run(['cargo', 'build', '--release'], cwd=REPO_DIR, check=True)
BINARY = REPO_DIR / 'target' / 'release' / 'taskgen'
if not BINARY.exists():
    raise FileNotFoundError(f'Binary tidak ditemukan: {BINARY}')

# Ambil daftar model yang mendukung generateContent
try:
    response = requests.get('https://generativelanguage.googleapis.com/v1beta/models',
        headers={'x-goog-api-key': API_KEY}, timeout=25)
    response.raise_for_status()
    model_rows = []
    for item in response.json().get('models', []):
        name = item.get('name', '').removeprefix('models/')
        if name and 'generateContent' in item.get('supportedGenerationMethods', []):
            model_rows.append((name, item.get('displayName', name)))
    model_rows.sort(key=lambda x: x[0])
except Exception as exc:
    print('Peringatan: daftar model tidak bisa dimuat:', type(exc).__name__)
    model_rows = []
if not model_rows:
    model_rows = [('gemini-3.1-flash-lite', 'gemini-3.1-flash-lite (fallback; cek ketersediaan)')]

# UI semua opsi tampil bersamaan
style = {'description_width': 'initial'}
model = widgets.Dropdown(options=[(f'{label} — {name}', name) for name, label in model_rows],
    value=next((n for n,_ in model_rows if n == 'gemini-3.1-flash-lite'), model_rows[0][0]),
    description='Model Gemini:', layout=widgets.Layout(width='100%'), style=style)
count = widgets.BoundedIntText(value=10, min=1, max=10000, description='Jumlah pertanyaan:', style=style, layout=widgets.Layout(width='48%'))
difficulty = widgets.Dropdown(options=[('Campuran (15% easy / 45% medium / 25% hard / 15% expert)', 'mixed'), ('Easy — dasar/satu langkah', 'easy'), ('Medium — penerapan/beberapa langkah', 'medium'), ('Hard — analisis multi-langkah', 'hard'), ('Expert — sintesis tingkat lanjut', 'expert')], value='medium', description='Kesulitan:', style=style, layout=widgets.Layout(width='48%'))
categories = widgets.SelectMultiple(options=[('Matematika','math'),('Pemrograman','coding'),('Sains','science'),('Ilmu komputer','cs'),('Kreativitas/Penulisan','creative'),('Percakapan','conversation')], value=('coding',), description='Kategori:', style=style, layout=widgets.Layout(width='100%', height='125px'))
workers = widgets.BoundedIntText(value=1, min=1, max=32, description='Workers:', style=style, layout=widgets.Layout(width='48%'))
dedup = widgets.Checkbox(value=True, description='Cegah duplikat teks di dataset gabungan', indent=False, layout=widgets.Layout(width='100%'))
run_button = widgets.Button(description='✨ Generate', button_style='success', icon='play', layout=widgets.Layout(width='200px', height='42px'))
status = widgets.Output()

print('\nAtur semuanya di bawah ini, lalu tekan Generate. Bahasa dikunci ke Indonesia (id).')
display(widgets.VBox([
    model,
    widgets.HBox([count, difficulty]),
    categories,
    workers,
    dedup,
    run_button,
    status,
]))

# Deduplikasi konservatif: teks prompt saja, normalisasi spasi/huruf kecil.
def prompt_key(row):
    prompt = row.get('prompt', '')
    normalized = re.sub(r'\s+', ' ', str(prompt)).strip().casefold()
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest() if normalized else None

def load_dataset():
    """Baca dataset utama; jangan diam-diam menghapus data jika JSON rusak."""
    if not DATASET_PATH.exists():
        return []
    with DATASET_PATH.open(encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f'{DATASET_PATH} harus berisi JSON array.')
    return data

def save_dataset_atomic(rows):
    """Tulis ke file sementara lalu ganti file utama secara atomik."""
    temp_path = DATASET_PATH.with_suffix('.json.tmp')
    with temp_path.open('w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
        f.write('\n')
        f.flush()
        os.fsync(f.fileno())
    temp_path.replace(DATASET_PATH)

def load_master_keys(existing_rows=None):
    rows = load_dataset() if existing_rows is None else existing_rows
    return {key for row in rows if (key := prompt_key(row))}

def on_generate(_):
    run_button.disabled = True
    temp_path = Path('/content/taskgen_current_run.jsonl')
    try:
        with status:
            clear_output(wait=True)
            if temp_path.exists():
                temp_path.unlink()
            selected_categories = list(categories.value)
            if not selected_categories:
                print('Pilih setidaknya satu kategori.')
                return

            weight = 1.0 / len(selected_categories)
            weights = [weight] * len(selected_categories)
            weights[-1] = 1.0 - sum(weights[:-1])
            distribution = ','.join(f'{cat}={w:.6f}' for cat, w in zip(selected_categories, weights))
            command = [str(BINARY), '--count', str(count.value), '--model', model.value,
                       '--distribution', distribution, '--workers', str(workers.value),
                       '--temperature', '1.0', '--output', str(temp_path)]
            if difficulty.value != 'mixed':
                command.extend(['--difficulty', f'{difficulty.value}=1.0'])

            print('Memulai generasi...')
            print('Kategori:', ', '.join(selected_categories), '| Bahasa: id | Model:', model.value)
            result = subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy())
            if not temp_path.exists():
                print('Generator tidak menghasilkan file. Exit code:', result.returncode)
                return

            rows, invalid = [], 0
            with temp_path.open(encoding='utf-8') as f:
                for line in f:
                    if not line.strip():
                        continue
                    try:
                        row = json.loads(line)
                        if isinstance(row, dict) and row.get('prompt'):
                            rows.append(row)
                        else:
                            invalid += 1
                    except json.JSONDecodeError:
                        invalid += 1

            if result.returncode != 0:
                print('Peringatan: generator berakhir dengan exit code non-zero; hasil valid yang ditemukan tetap akan diproses.')

            # Preserve existing records, migrate their schema, then allocate globally unique IDs.
            dataset = load_dataset()
            existing_keys = load_master_keys(dataset) if dedup.value else set()
            run_seen = set()
            added, duplicate_count = [], 0

            def normalize_difficulty(value):
                if isinstance(value, (int, float)) and not isinstance(value, bool):
                    number = int(value)
                    if number <= 3: return 'easy'
                    if number <= 6: return 'medium'
                    if number <= 8: return 'hard'
                    return 'expert'
                label = str(value or 'medium').strip().lower()
                return label if label in {'easy', 'medium', 'hard', 'expert'} else 'medium'

            clean_dataset = []
            max_id = 0
            seen_ids = set()
            for old_row in dataset:
                if isinstance(old_row, dict) and old_row.get('prompt'):
                    old_id = str(old_row.get('id') or '')
                    match = re.fullmatch(r'question-(\d+)', old_id)
                    if match:
                        max_id = max(max_id, int(match.group(1)))
                    clean_dataset.append({
                        'id': old_id,
                        'generated_date': old_row.get('generated_date'),
                        'domain': old_row.get('domain', ''),
                        'subdomain': old_row.get('subdomain', ''),
                        'difficulty': normalize_difficulty(old_row.get('difficulty')),
                        'language': old_row.get('language') or 'id',
                        'prompt': str(old_row.get('prompt', '')).strip(),
                        'model': old_row.get('model') or old_row.get('taskgen_model') or model.value,
                    })

            for old_row in clean_dataset:
                if not old_row['id'] or old_row['id'] in seen_ids:
                    while True:
                        max_id += 1
                        candidate = f'question-{max_id:06d}'
                        if candidate not in seen_ids:
                            old_row['id'] = candidate
                            break
                seen_ids.add(old_row['id'])

            existing_keys = {key for row in clean_dataset if (key := prompt_key(row))} if dedup.value else set()
            run_seen = set()
            for row in rows:
                key = prompt_key(row)
                if dedup.value and key and (key in existing_keys or key in run_seen):
                    duplicate_count += 1
                    continue
                max_id += 1
                added.append({
                    'id': f'question-{max_id:06d}',
                    'generated_date': row.get('generated_date') or time.strftime('%Y-%m-%d'),
                    'domain': row.get('domain', ''),
                    'subdomain': row.get('subdomain', ''),
                    'difficulty': normalize_difficulty(row.get('difficulty')),
                    'language': row.get('language') or 'id',
                    'prompt': str(row.get('prompt', '')).strip(),
                    'model': row.get('model') or row.get('taskgen_model') or model.value,
                })
                if key:
                    run_seen.add(key)

            # Persist schema migration even if this run only produced duplicates.
            save_dataset_atomic(clean_dataset + added)

            print('\nRun selesai.')
            print('Baris valid yang dihasilkan:', len(rows))
            print('Baris tidak valid:', invalid)
            print('Duplikat yang dilewati:', duplicate_count)
            print('Baris baru ditambahkan:', len(added))
            print('Total pertanyaan dalam dataset:', len(dataset) + len(added))
            print('Dataset tunggal:', DATASET_PATH)
            print('Format setiap task: id, generated_date, domain, subdomain, difficulty, language, prompt, model.')
    except Exception as exc:
        with status:
            print('Terjadi error:', type(exc).__name__, str(exc))
            print('Dataset utama tidak ditimpa jika pembacaan atau penyimpanan gagal.')
    finally:
        if temp_path.exists():
            temp_path.unlink()
        run_button.disabled = False

run_button.on_click(on_generate)

Menghubungkan Google Drive...
Mounted at /content/drive
Mengunduh TaskGen dari branch: main
Memasang Rust toolchain...
Membangun TaskGen (pertama kali bisa memerlukan beberapa menit)...

Atur semuanya di bawah ini, lalu tekan Generate. Bahasa dikunci ke Indonesia (id).
